# Análisis de Atención en Transformers — La Metamorfosis (Kafka)

Modelo: `bert-base-multilingual-cased`. Corpus: 2 páginas consecutivas al azar de *La Metamorfosis*.

In [ ]:
!pip install -q transformers torch pandas

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
modelo = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
modelo.eval()

## Corpus

Cargar las 2 páginas consecutivas seleccionadas y las oraciones de interés.

In [ ]:
with open("../data/corpus/pagina_1.txt", encoding="utf-8") as f:
    pagina_1 = f.read()
with open("../data/corpus/pagina_2.txt", encoding="utf-8") as f:
    pagina_2 = f.read()

# Oraciones seleccionadas para el análisis (llenar tras leer las páginas)
oraciones = [
    "",
    "",
]
oraciones

## Parte A: Tokenización

Tokens especiales, subpalabras, diferencias entre palabras lingüísticas y tokens del modelo.

In [ ]:
def mostrar_tokens(oracion: str) -> list[str]:
    entradas = tokenizer(oracion, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(entradas["input_ids"][0])
    print(oracion)
    print(tokens)
    return tokens

for oracion in oraciones:
    mostrar_tokens(oracion)

## Parte B: Ejecución del modelo

Correr el modelo solicitando atenciones y reportar la forma de las matrices.

In [ ]:
def obtener_attentions(oracion: str):
    entradas = tokenizer(oracion, return_tensors="pt")
    with torch.no_grad():
        outputs = modelo(**entradas)
    attentions = outputs.attentions  # tuple(num_capas) de (batch, num_cabezas, num_tokens, num_tokens)
    tokens = tokenizer.convert_ids_to_tokens(entradas["input_ids"][0])
    return attentions, tokens

attentions, tokens = obtener_attentions(oraciones[0])
print("num capas:", len(attentions))
print("shape por capa:", attentions[0].shape)

## Parte C: Análisis de atención

Para 2 capas x 2 cabezas x 2 tokens relevantes por oración, mostrar los 5 tokens con mayor peso de atención.

In [ ]:
def top_atencion(attentions, tokens, capa: int, cabeza: int, token_idx: int, top_k: int = 5) -> pd.DataFrame:
    pesos = attentions[capa][0, cabeza, token_idx]
    top_vals, top_idx = torch.topk(pesos, top_k)
    return pd.DataFrame({
        "token_objetivo": [tokens[token_idx]] * top_k,
        "token_atendido": [tokens[i] for i in top_idx.tolist()],
        "peso": top_vals.tolist(),
        "capa": capa,
        "cabeza": cabeza,
    })

# Ejemplo: llenar capas, cabezas y token_idx relevantes tras inspeccionar `tokens`
# top_atencion(attentions, tokens, capa=0, cabeza=0, token_idx=1)

## Parte D: Comparación entre oraciones

Comparar al menos dos oraciones y explicar si cambia la atención cuando cambia el contexto.

## Preguntas de análisis

1. ¿Qué tokens reciben mayor atención desde cada token seleccionado?
2. ¿Cambian los patrones entre capas?
3. ¿Cambian los patrones entre cabezas?
4. ¿Las palabras con mayor atención son lingüísticamente relevantes?
5. ¿Qué diferencias aparecen entre oraciones simples y complejas?
6. ¿Qué ocurre cuando una palabra se divide en subpalabras?
7. ¿Qué no puedes concluir observando únicamente los pesos de atención?

_(Respuestas a completar tras el análisis.)_